# 🦕 DINOv2 Self-Supervised Feature Extraction for SylFishBD

**Assignment 2: Self-Supervised Learning - Phase 1: Feature Extraction**

This notebook demonstrates using **DINOv2** (Self-Distillation with No Labels v2) for self-supervised feature extraction on the SylFishBD dataset.

---

## 📚 Table of Contents

1. [Introduction & Theory](#1-introduction--theory)
2. [Environment Setup](#2-environment-setup)
3. [Configuration](#3-configuration)
4. [Load DINOv2 Model](#4-load-dinov2-model)
5. [Data Loading](#5-data-loading)
6. [Feature Extraction](#6-feature-extraction)
7. [Feature Analysis](#7-feature-analysis)
8. [Feature Visualization](#8-feature-visualization)
9. [Save Features](#9-save-features)
10. [Summary](#10-summary)

---

## 1. Introduction & Theory

### What is DINOv2?

DINOv2 (Distillation with No Labels v2) is a self-supervised Vision Transformer developed by Meta AI. It learns powerful visual representations from images **without any labels** through self-distillation.

### Key Innovations:

1. **Self-Distillation**: Student network learns from a teacher (momentum encoder)
2. **Multi-Crop Strategy**: Processes global and local views of images
3. **No Labels Required**: Pure self-supervised learning
4. **Semantic Features**: Learns semantically meaningful representations

### DINOv2 Architecture:

```
                    Input Image

                        │

         ┌──────────────┴──────────────┐

         ▼                             ▼

   ┌───────────┐                 ┌───────────┐

   │  Global   │                 │  Local    │

   │  Views    │                 │  Views    │

   │ (224×224) │                 │ (96×96)   │

   └─────┬─────┘                 └─────┬─────┘

         │                             │

         └──────────────┬──────────────┘

                        ▼

              ┌─────────────────┐

              │  Vision         │

              │  Transformer    │

              │  (ViT)          │

              └────────┬────────┘

                       │

                       ▼

              ┌─────────────────┐

              │  [CLS] Token    │

              │  Feature (384d) │

              └─────────────────┘

```

### Available DINOv2 Models:

| Model | Parameters | Feature Dim | Best For |
|-------|-----------|-------------|----------|
| dinov2_vits14 | 22M | 384 | Fast inference |
| dinov2_vitb14 | 86M | 768 | Balanced |
| dinov2_vitl14 | 300M | 1024 | High accuracy |
| dinov2_vitg14 | 1.1B | 1536 | Best accuracy |

### Why Use Pre-trained DINOv2?

Training DINOv2 from scratch requires:
- 142 million curated images
- Massive compute resources (16 A100 GPUs for days)
- Careful hyperparameter tuning

**Instead, we use the pre-trained model to extract features for our downstream tasks!**

### What This Notebook Produces:

- **Feature files**: Extracted DINOv2 features for train/val/test
- **Visualizations**: t-SNE/PCA of feature space
- **Analysis**: Feature statistics and quality assessment

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

# Sklearn
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Seaborn
try:
    import seaborn as sns
    sns.set_style('whitegrid')
    HAS_SEABORN = True
except:
    HAS_SEABORN = False

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

# ==================================
# DEVICE SELECTION (CUDA > MPS > CPU)
# ==================================
def get_device():
    """Get the best available device."""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        device_name = torch.cuda.get_device_name(0)
        memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"🖥️  Device: CUDA")
        print(f"   GPU: {device_name}")
        print(f"   Memory: {memory:.2f} GB")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        print(f"🖥️  Device: MPS (Apple Silicon)")
    else:
        device = torch.device('cpu')
        print(f"🖥️  Device: CPU")
    return device

device = get_device()

# Environment detection
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

print(f"📍 Environment: {'🌐 Kaggle' if IS_KAGGLE else '🔬 Colab' if IS_COLAB else '💻 Local'}")
print(f"🐍 PyTorch Version: {torch.__version__}")
print("✅ All imports successful!")

In [ ]:
class Config:
    """DINOv2 Feature Extraction Configuration"""
    
    # ==================================
    # PATH CONFIGURATION
    # ==================================
    DATA_DIR = Path('sylfishbd')
    OUTPUT_DIR = Path('outputs/04_1_DINOv2_FeatureExtraction_SylFishBD')
    
    # DINOv2 Model Configuration
    MODEL_NAME = 'dinov2_vits14'  # Options: dinov2_vits14, dinov2_vitb14, dinov2_vitl14
    FEATURE_DIM = 384  # ViT-Small dimension (384 for vits14, 768 for vitb14, 1024 for vitl14)
    
    # Image settings
    IMG_SIZE = 224  # DINOv2 expects 224x224 for patch size 14
    BATCH_SIZE = 32
    
    # Classes for SylFishBD dataset
    CLASS_NAMES = ['boal', 'ilish', 'kalibaush', 'katla', 'koi', 'mrigel', 'pabda', 'rui', 'telapia']
    NUM_CLASSES = 9
    
    # Output file naming
    FEATURES_SAVE_PREFIX = 'dinov2_features'

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==================================
# PRINT CONFIGURATION & VERIFY PATHS
# ==================================
print("=" * 70)
print("📋 DINOv2 FEATURE EXTRACTION CONFIGURATION")
print("=" * 70)
print(f"  📂 Data Directory:  {config.DATA_DIR}")
print(f"  📁 Output Directory: {config.OUTPUT_DIR}")
print(f"  🤖 Model:           {config.MODEL_NAME}")
print(f"  📐 Feature Dim:     {config.FEATURE_DIM}")
print(f"  🖼️  Image Size:      {config.IMG_SIZE}x{config.IMG_SIZE}")
print(f"  📦 Batch Size:      {config.BATCH_SIZE}")
print(f"  🏷️  Classes:         {config.CLASS_NAMES}")
print("=" * 70)

# Verify data directory exists
if config.DATA_DIR.exists():
    print(f"\n✅ Data directory found!")
    # Load metadata
    metadata_path = config.DATA_DIR / 'metadata.csv'
    if metadata_path.exists():
        metadata = pd.read_csv(metadata_path)
        print(f"   📊 Metadata loaded: {len(metadata)} images")
        print(f"   🏷️  Unique classes: {metadata['class_name'].nunique()}")
    else:
        print(f"   ❌ metadata.csv not found")
else:
    print(f"\n❌ ERROR: Data directory not found at {config.DATA_DIR}")
    print("   Please check the DATA_DIR path.")

In [ ]:
print(f"🔄 Loading DINOv2 model: {config.MODEL_NAME}")
print("   This may take a moment on first run (downloading weights)...")

try:
    # Load from torch hub
    dinov2_model = torch.hub.load('facebookresearch/dinov2', config.MODEL_NAME)
    dinov2_model = dinov2_model.to(device)
    dinov2_model.eval()  # Set to evaluation mode
    
    print(f"\n✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("   Trying alternative method...")
    dinov2_model = torch.hub.load('facebookresearch/dinov2', config.MODEL_NAME, pretrained=True)
    dinov2_model = dinov2_model.to(device)
    dinov2_model.eval()

# Model info
total_params = sum(p.numel() for p in dinov2_model.parameters())
print(f"\n📊 Model Statistics:")
print(f"   Parameters: {total_params:,}")
print(f"   Feature Dimension: {config.FEATURE_DIM}")

In [ ]:
# Test with dummy input
print("\n🧪 Testing model with dummy input...")
with torch.no_grad():
    dummy_input = torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE).to(device)
    dummy_output = dinov2_model(dummy_input)
    print(f"   Input shape: {dummy_input.shape}")
    print(f"   Output shape: {dummy_output.shape}")
    print(f"   ✅ Model working correctly!")

In [ ]:
# Load metadata and create splits
metadata = pd.read_csv(config.DATA_DIR / 'metadata.csv')

# Map class names to IDs
class_to_id = {name: idx for idx, name in enumerate(config.CLASS_NAMES)}
metadata['label'] = metadata['class_name'].map(class_to_id)

# Split data: 70% train, 15% val, 15% test (stratified)
train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"📊 Data splits:")
print(f"   Train: {len(train_df)} images")
print(f"   Validation: {len(val_df)} images")
print(f"   Test: {len(test_df)} images")

# Dataset class for SylFishBD
class SylFishBDDataset(Dataset):
    def __init__(self, df, img_base_dir, transform=None):
        self.df = df
        self.img_base_dir = Path(img_base_dir)
        self.transform = transform
        
        print(f"📁 Loaded {len(df)} images from SylFishBD dataset")
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_base_dir / row['image_path']
        image = Image.open(img_path).convert('RGB')
        label = row['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, str(img_path)

# DINOv2-specific transforms
dinov2_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = SylFishBDDataset(train_df, config.DATA_DIR, transform=dinov2_transform)
val_dataset = SylFishBDDataset(val_df, config.DATA_DIR, transform=dinov2_transform)
test_dataset = SylFishBDDataset(test_df, config.DATA_DIR, transform=dinov2_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n📊 Dataset Summary:")
print(f"   Train: {len(train_dataset)} images")
print(f"   Validation: {len(val_dataset)} images")
print(f"   Test: {len(test_dataset)} images")

In [ ]:
# Visualize sample images
def show_sample_images(dataset, num_samples=6):
    """Display sample images from the dataset."""
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    for idx, ax in zip(indices, axes):
        img, label, path = dataset[idx]
        
        # Denormalize
        img_np = img.numpy().transpose(1, 2, 0) * std + mean
        img_np = np.clip(img_np, 0, 1)
        
        ax.imshow(img_np)
        ax.set_title(f"Class: {config.CLASS_NAMES[label]}", fontsize=11)
        ax.axis('off')
    
    plt.suptitle('Sample Images for DINOv2 Feature Extraction', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'dinov2_sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved: {config.OUTPUT_DIR / 'dinov2_sample_images.png'}")

show_sample_images(train_dataset)

### 5.1 Exploratory Data Analysis (EDA)

Before feature extraction, let's analyze the dataset to understand its characteristics.

In [ ]:
# ==================================
# COMPREHENSIVE EDA
# ==================================
print("=" * 70)
print("📊 EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# Collect dataset statistics
def get_dataset_stats(dataset, name):
    """Collect statistics from a dataset."""
    labels = []
    img_sizes = []
    aspects = []
    
    for i in range(min(len(dataset), 200)):  # Sample for speed
        _, label, path = dataset[i]
        labels.append(label)
        
        # Get original image dimensions
        try:
            img = Image.open(path)
            w, h = img.size
            img_sizes.append((w, h))
            aspects.append(w / h if h > 0 else 1)
        except:
            pass
    
    return labels, img_sizes, aspects

train_labels_eda, train_sizes, train_aspects = get_dataset_stats(train_dataset, 'Train')
val_labels_eda, _, _ = get_dataset_stats(val_dataset, 'Val')
test_labels_eda, _, _ = get_dataset_stats(test_dataset, 'Test')

# Create comprehensive EDA plots
fig = plt.figure(figsize=(16, 12))

# 1. Class Distribution Bar Chart
ax1 = fig.add_subplot(2, 3, 1)
splits = ['Train', 'Validation', 'Test']
all_labels = [train_labels_eda, val_labels_eda, test_labels_eda]
x = np.arange(config.NUM_CLASSES)
width = 0.25
colors = ['#3498db', '#2ecc71', '#e74c3c']

for i, (split, labels, color) in enumerate(zip(splits, all_labels, colors)):
    counts = [labels.count(c) for c in range(config.NUM_CLASSES)]
    ax1.bar(x + i * width, counts, width, label=split, color=color, edgecolor='black')

ax1.set_xlabel('Class', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title('Class Distribution Across Splits', fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(config.CLASS_NAMES)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Pie Chart for Training Set
ax2 = fig.add_subplot(2, 3, 2)
train_counts = [train_labels_eda.count(c) for c in range(config.NUM_CLASSES)]
explode = [0.02] * config.NUM_CLASSES
colors_pie = plt.cm.Set2(np.linspace(0, 1, config.NUM_CLASSES))
wedges, texts, autotexts = ax2.pie(train_counts, labels=config.CLASS_NAMES, 
                                    autopct='%1.1f%%', explode=explode,
                                    colors=colors_pie, startangle=90)
ax2.set_title('Training Set Class Balance', fontweight='bold')

# 3. Image Size Distribution
ax3 = fig.add_subplot(2, 3, 3)
if train_sizes:
    widths = [s[0] for s in train_sizes]
    heights = [s[1] for s in train_sizes]
    ax3.scatter(widths, heights, alpha=0.5, c='steelblue', s=30)
    ax3.axhline(y=np.mean(heights), color='r', linestyle='--', label=f'Mean H: {np.mean(heights):.0f}')
    ax3.axvline(x=np.mean(widths), color='g', linestyle='--', label=f'Mean W: {np.mean(widths):.0f}')
ax3.set_xlabel('Width (pixels)', fontsize=11)
ax3.set_ylabel('Height (pixels)', fontsize=11)
ax3.set_title('Original Image Dimensions', fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# 4. Aspect Ratio Distribution
ax4 = fig.add_subplot(2, 3, 4)
if train_aspects:
    ax4.hist(train_aspects, bins=30, color='coral', edgecolor='black', alpha=0.7)
    ax4.axvline(x=1.0, color='r', linestyle='--', linewidth=2, label='Square (1:1)')
    ax4.axvline(x=np.mean(train_aspects), color='g', linestyle='--', label=f'Mean: {np.mean(train_aspects):.2f}')
ax4.set_xlabel('Aspect Ratio (W/H)', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('Aspect Ratio Distribution', fontweight='bold')
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

# 5. Samples per Split
ax5 = fig.add_subplot(2, 3, 5)
split_sizes = [len(train_dataset), len(val_dataset), len(test_dataset)]
bars = ax5.bar(['Train', 'Validation', 'Test'], split_sizes, 
               color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='black')
for bar, size in zip(bars, split_sizes):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{size}', ha='center', va='bottom', fontweight='bold')
ax5.set_ylabel('Number of Images', fontsize=11)
ax5.set_title('Dataset Split Sizes', fontweight='bold')
ax5.grid(axis='y', alpha=0.3)

# 6. Split Proportions
ax6 = fig.add_subplot(2, 3, 6)
total = sum(split_sizes)
proportions = [s/total*100 for s in split_sizes]
ax6.barh(['Train', 'Validation', 'Test'], proportions, 
         color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='black')
for i, (prop, size) in enumerate(zip(proportions, split_sizes)):
    ax6.text(prop + 1, i, f'{prop:.1f}%', va='center', fontweight='bold')
ax6.set_xlabel('Percentage (%)', fontsize=11)
ax6.set_title('Dataset Split Proportions', fontweight='bold')
ax6.set_xlim(0, 100)
ax6.grid(axis='x', alpha=0.3)

plt.suptitle('DINOv2 Feature Extraction - Dataset EDA', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'dinov2_dataset_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Saved: {config.OUTPUT_DIR / 'dinov2_dataset_eda.png'}")

# Print summary statistics
print(f"\n📊 DATASET SUMMARY:")
print(f"   Total Images: {total}")
print(f"   Train: {len(train_dataset)} ({len(train_dataset)/total*100:.1f}%)")
print(f"   Validation: {len(val_dataset)} ({len(val_dataset)/total*100:.1f}%)")
print(f"   Test: {len(test_dataset)} ({len(test_dataset)/total*100:.1f}%)")

In [ ]:
# Feature extraction function
@torch.no_grad()
def extract_dinov2_features(model, dataloader, desc="Extracting features"):
    """
    Extract DINOv2 CLS token features from all images.
    
    Returns:
        features: numpy array of shape (N, feature_dim)
        labels: numpy array of labels
        paths: list of image paths
    """
    model.eval()
    
    all_features = []
    all_labels = []
    all_paths = []
    
    for images, labels, paths in tqdm(dataloader, desc=desc):
        images = images.to(device)
        
        # Extract CLS token features
        features = model(images)
        
        all_features.append(features.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_paths.extend(paths)
    
    features = np.concatenate(all_features, axis=0)
    labels = np.array(all_labels)
    
    return features, labels, all_paths

print("🔍 Extracting DINOv2 features...\n")

# Extract from all splits
train_features, train_labels, train_paths = extract_dinov2_features(
    dinov2_model, train_loader, desc="Training set"
)

val_features, val_labels, val_paths = extract_dinov2_features(
    dinov2_model, val_loader, desc="Validation set"
)

test_features, test_labels, test_paths = extract_dinov2_features(
    dinov2_model, test_loader, desc="Test set"
)

print(f"\n📊 Extracted Features:")
print(f"   Train: {train_features.shape}")
print(f"   Validation: {val_features.shape}")
print(f"   Test: {test_features.shape}")

In [ ]:
# Feature analysis
def analyze_features(features, labels, class_names, split_name):
    """
    Analyze feature statistics.
    """
    print(f"\n{'='*60}")
    print(f"📊 FEATURE ANALYSIS: {split_name}")
    print(f"{'='*60}")
    
    print(f"\n📐 Overall Statistics:")
    print(f"   Shape: {features.shape}")
    print(f"   Mean: {features.mean():.4f}")
    print(f"   Std: {features.std():.4f}")
    print(f"   Min: {features.min():.4f}")
    print(f"   Max: {features.max():.4f}")
    
    # L2 norms
    norms = np.linalg.norm(features, axis=1)
    print(f"\n📏 L2 Norm Statistics:")
    print(f"   Mean: {norms.mean():.4f}")
    print(f"   Std: {norms.std():.4f}")
    
    # Per-class analysis
    print(f"\n🏷️ Per-Class Statistics:")
    for cls_id in range(len(class_names)):
        mask = labels == cls_id
        if mask.sum() > 0:
            cls_features = features[mask]
            cls_norms = norms[mask]
            print(f"   {class_names[cls_id]}:")
            print(f"      Count: {mask.sum()}")
            print(f"      Mean L2 Norm: {cls_norms.mean():.4f}")
            print(f"      Feature Mean: {cls_features.mean():.4f}")

# Analyze training features
analyze_features(train_features, train_labels, config.CLASS_NAMES, "Training Set")

In [ ]:
# Feature distribution plot
def plot_feature_distributions(features, labels, class_names):
    """
    Plot feature value distributions.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Overall feature distribution
    axes[0].hist(features.flatten(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Feature Value')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Overall Feature Distribution', fontweight='bold')
    axes[0].axvline(features.mean(), color='red', linestyle='--', label=f'Mean: {features.mean():.2f}')
    axes[0].legend()
    
    # L2 norm distribution
    norms = np.linalg.norm(features, axis=1)
    axes[1].hist(norms, bins=30, alpha=0.7, color='coral', edgecolor='black')
    axes[1].set_xlabel('L2 Norm')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Feature L2 Norm Distribution', fontweight='bold')
    axes[1].axvline(norms.mean(), color='red', linestyle='--', label=f'Mean: {norms.mean():.2f}')
    axes[1].legend()
    
    # Per-class norm distribution
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    for cls_id in range(len(class_names)):
        mask = labels == cls_id
        if mask.sum() > 0:
            cls_norms = norms[mask]
            axes[2].hist(cls_norms, bins=20, alpha=0.5, label=class_names[cls_id], color=colors[cls_id])
    axes[2].set_xlabel('L2 Norm')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('L2 Norm by Class', fontweight='bold')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'dinov2_feature_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved: {config.OUTPUT_DIR / 'dinov2_feature_distributions.png'}")

plot_feature_distributions(train_features, train_labels, config.CLASS_NAMES)

### Understanding Feature Distributions:

**Feature Values (Left):**

- Shows the distribution of individual feature values
- Should be roughly centered around 0 for well-normalized features

**L2 Norms (Center):**

- Measures the "magnitude" of each feature vector
- Similar norms suggest consistent feature extraction

**Per-Class L2 Norms (Right):**

- Compares norm distributions across classes
- Large differences might indicate class-specific characteristics

In [ ]:
# Feature visualization
def visualize_features(features, labels, class_names, method='tsne', title='DINOv2 Features'):
    """
    Visualize features in 2D.
    """
    print(f"\n🎨 Computing {method.upper()} embedding...")
    
    if method == 'tsne':
        perplexity = min(30, len(features) - 1)
        reducer = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    else:  # PCA
        reducer = PCA(n_components=2, random_state=42)
    
    embeddings = reducer.fit_transform(features)
    
    # Plot
    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    
    for cls_id in range(len(class_names)):
        mask = labels == cls_id
        plt.scatter(embeddings[mask, 0], embeddings[mask, 1],
                   c=[colors[cls_id]], label=class_names[cls_id],
                   alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
    
    plt.xlabel(f'{method.upper()} Dimension 1', fontsize=12)
    plt.ylabel(f'{method.upper()} Dimension 2', fontsize=12)
    plt.title(f'{title} ({method.upper()})', fontsize=14, fontweight='bold')
    plt.legend(title='Classes', loc='best', fontsize=11)
    plt.grid(True, alpha=0.3)
    
    filename = f"dinov2_{method}_visualization.png"
    plt.savefig(config.OUTPUT_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved: {config.OUTPUT_DIR / filename}")
    
    return embeddings

# t-SNE visualization
tsne_embeddings = visualize_features(
    train_features, train_labels, config.CLASS_NAMES,
    method='tsne', title='DINOv2 Training Features'
)

# PCA visualization
pca_embeddings = visualize_features(
    train_features, train_labels, config.CLASS_NAMES,
    method='pca', title='DINOv2 Training Features'
)

### Understanding Feature Visualizations:

**What to Look For:**

- **Clear Clusters**: Good separation between classes indicates meaningful features
- **Tight Clusters**: Features are consistent within each class
- **Overlap**: May indicate challenging samples or similar classes

**t-SNE vs PCA:**

- **t-SNE**: Better for revealing local structure and clusters
- **PCA**: Shows global variance structure; explains how much variance is captured

**Good Signs:**

- Different classes form distinct groups
- Points of the same class are close together

**DINOv2 features typically show good clustering even without task-specific training!**

In [ ]:
# Compare train and test feature distributions
def compare_feature_space(train_feat, train_lbl, test_feat, test_lbl, class_names):
    """
    Compare train and test feature distributions using PCA.
    """
    # Combine for joint PCA
    combined = np.vstack([train_feat, test_feat])
    
    pca = PCA(n_components=2, random_state=42)
    combined_2d = pca.fit_transform(combined)
    
    train_2d = combined_2d[:len(train_feat)]
    test_2d = combined_2d[len(train_feat):]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    
    # Training
    for cls_id in range(len(class_names)):
        mask = train_lbl == cls_id
        axes[0].scatter(train_2d[mask, 0], train_2d[mask, 1],
                       c=[colors[cls_id]], label=class_names[cls_id],
                       alpha=0.7, s=50)
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].set_title('Training Set Features', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Test
    for cls_id in range(len(class_names)):
        mask = test_lbl == cls_id
        axes[1].scatter(test_2d[mask, 0], test_2d[mask, 1],
                       c=[colors[cls_id]], label=class_names[cls_id],
                       alpha=0.7, s=50)
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')
    axes[1].set_title('Test Set Features', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(f'DINOv2 Feature Space Comparison (PCA, Var: {sum(pca.explained_variance_ratio_)*100:.1f}%)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'dinov2_train_test_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved: {config.OUTPUT_DIR / 'dinov2_train_test_comparison.png'}")

compare_feature_space(train_features, train_labels, test_features, test_labels, config.CLASS_NAMES)

In [ ]:
# Save features
def save_features(features, labels, paths, split_name, output_dir, prefix):
    """
    Save features, labels, and paths to files.
    These will be loaded by the fine-tuning notebook.
    """
    # Save as numpy arrays
    feat_path = output_dir / f'{prefix}_{split_name}_features.npy'
    lbl_path = output_dir / f'{prefix}_{split_name}_labels.npy'
    
    np.save(feat_path, features)
    np.save(lbl_path, labels)
    
    # Save paths as CSV for reference
    paths_df = pd.DataFrame({
        'path': paths, 
        'label': labels,
        'class_name': [config.CLASS_NAMES[l] for l in labels]
    })
    paths_df.to_csv(output_dir / f'{prefix}_{split_name}_paths.csv', index=False)
    
    print(f"   ✅ {split_name.upper()}: {features.shape}")
    print(f"      Features: {feat_path.name}")
    print(f"      Labels:   {lbl_path.name}")

print("=" * 70)
print("💾 SAVING EXTRACTED FEATURES")
print("=" * 70)
print(f"\n📁 Output directory: {config.OUTPUT_DIR}\n")

save_features(train_features, train_labels, train_paths, 'train', 
              config.OUTPUT_DIR, config.FEATURES_SAVE_PREFIX)
save_features(val_features, val_labels, val_paths, 'val', 
              config.OUTPUT_DIR, config.FEATURES_SAVE_PREFIX)
save_features(test_features, test_labels, test_paths, 'test', 
              config.OUTPUT_DIR, config.FEATURES_SAVE_PREFIX)

# Save configuration for reference
config_dict = {
    'model_name': config.MODEL_NAME,
    'feature_dim': config.FEATURE_DIM,
    'img_size': config.IMG_SIZE,
    'class_names': config.CLASS_NAMES,
    'num_classes': config.NUM_CLASSES,
    'train_samples': len(train_features),
    'val_samples': len(val_features),
    'test_samples': len(test_features),
    'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

config_df = pd.DataFrame([config_dict])
config_df.to_csv(config.OUTPUT_DIR / 'dinov2_extraction_config.csv', index=False)

print(f"\n📋 Configuration saved to: dinov2_extraction_config.csv")

# List all saved files
print(f"\n📁 ALL SAVED FILES:")
for f in sorted(config.OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name} ({size_kb:.1f} KB)")

In [ ]:
# Class distribution in features
def plot_class_distribution(labels, class_names, title):
    counts = Counter(labels)
    
    plt.figure(figsize=(8, 5))
    bars = plt.bar(class_names, [counts[i] for i in range(len(class_names))],
                   color=plt.cm.Set2(np.linspace(0, 1, len(class_names))),
                   edgecolor='black')
    
    for bar, count in zip(bars, [counts[i] for i in range(len(class_names))]):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title(title, fontweight='bold')
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'dinov2_class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_class_distribution(train_labels, config.CLASS_NAMES, 'Training Set Class Distribution')

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("📋 DINOv2 FEATURE EXTRACTION - COMPLETE SUMMARY")
print("=" * 70)
print(f"""
🤖 MODEL INFORMATION:

   ├── Model: {config.MODEL_NAME}
   ├── Parameters: {sum(p.numel() for p in dinov2_model.parameters()):,}
   └── Feature Dimension: {config.FEATURE_DIM}



📊 EXTRACTED FEATURES:

   ┌─────────────┬────────────────────┬──────────────┐
   │   Split     │      Shape         │   Samples    │
   ├─────────────┼────────────────────┼──────────────┤
   │   Train     │ {str(train_features.shape):18s} │ {len(train_features):>10,d}   │
   │   Val       │ {str(val_features.shape):18s} │ {len(val_features):>10,d}   │
   │   Test      │ {str(test_features.shape):18s} │ {len(test_features):>10,d}   │
   └─────────────┴────────────────────┴──────────────┘



📈 FEATURE STATISTICS:

   ├── Mean: {train_features.mean():.4f}
   ├── Std:  {train_features.std():.4f}
   └── Mean L2 Norm: {np.linalg.norm(train_features, axis=1).mean():.4f}



💾 OUTPUT FILES (in {config.OUTPUT_DIR}):

   ├── {config.FEATURES_SAVE_PREFIX}_train_features.npy
   ├── {config.FEATURES_SAVE_PREFIX}_train_labels.npy
   ├── {config.FEATURES_SAVE_PREFIX}_val_features.npy
   ├── {config.FEATURES_SAVE_PREFIX}_val_labels.npy
   ├── {config.FEATURES_SAVE_PREFIX}_test_features.npy
   ├── {config.FEATURES_SAVE_PREFIX}_test_labels.npy
   ├── dinov2_extraction_config.csv
   └── [visualization images]



🚀 NEXT STEPS:

   1. Open '04_2_DINOv2_Finetuning.ipynb'
   2. Update FEATURES_DIR to point to: {config.OUTPUT_DIR}
   3. Run the fine-tuning notebook to train classifiers
   
   On Kaggle: Features will be in /kaggle/working/04_1_DINOv2_FeatureExtraction/

""")
print("=" * 70)